# M00 Zoom In: An Introduction to Circuits

Run a live visual-circuits reproduction on a public InceptionV1 model.


## What this notebook does

- Loads a public pretrained vision model.
- Generates feature visualizations with activation maximization.
- Validates one neuron on real CIFAR-10 images.
- Measures orientation tuning on synthetic arc stimuli.
- Traces one small circuit by following learned weights upstream.


In [ ]:
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "torchvision", "torch-lucent", "matplotlib", "numpy", "Pillow"],
        check=True,
    )

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from lucent.modelzoo import inceptionv1
from lucent.optvis import objectives, render
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader
from torchvision import transforms as T

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)

model = inceptionv1(pretrained=True).to(device).eval()
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]
transform = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize(imagenet_mean, imagenet_std),
])
render_cache = {}

def denormalize(image_tensor):
    mean = torch.tensor(imagenet_mean).view(3, 1, 1)
    std = torch.tensor(imagenet_std).view(3, 1, 1)
    restored = (image_tensor.cpu() * std + mean).clamp(0.0, 1.0)
    return restored.permute(1, 2, 0).numpy()

def layer_mean_activation(layer_name, channel, batch):
    capture = {}

    def hook_fn(_module, _inputs, output):
        capture["value"] = output[:, channel].mean(dim=(1, 2)).detach().cpu()

    hook = dict(model.named_modules())[layer_name].register_forward_hook(hook_fn)
    with torch.no_grad():
        model(batch.to(device))
    hook.remove()
    return capture["value"]

def render_neuron(layer_name, channel, steps=12):
    key = (layer_name, channel, steps)
    if key not in render_cache:
        images = render.render_vis(
            model,
            objectives.channel(layer_name, channel),
            thresholds=(steps,),
            show_inline=False,
            progress=False,
        )
        render_cache[key] = images[0][0]
    return render_cache[key]

print("Loaded InceptionV1 on", device)


In [ ]:
representative_neurons = [
    ("conv2d0", 0, "early edge / color"),
    ("mixed3a", 0, "edge bundle"),
    ("mixed3b", 379, "curve detector"),
]

fig, axes = plt.subplots(1, len(representative_neurons), figsize=(11, 3.6))
for ax, (layer_name, channel, label) in zip(axes, representative_neurons):
    image = render_neuron(layer_name, channel, steps=12)
    ax.imshow(image)
    ax.axis("off")
    ax.set_title(f"{layer_name}:{channel}\n{label}", fontsize=9)
plt.suptitle("Live feature visualizations from a pretrained vision model", fontsize=11)
plt.tight_layout()

noise_batch = torch.randn(8, 3, 224, 224)
print("Mean activation on random noise:")
for layer_name, channel, label in representative_neurons:
    score = float(layer_mean_activation(layer_name, channel, noise_batch).mean())
    print(f"  {layer_name}:{channel:>3} ({label}) -> {score:.4f}")


In [ ]:
data_root = Path.cwd() / ".learn_interpretability_data" / "cifar10"
dataset = torchvision.datasets.CIFAR10(
    data_root,
    train=False,
    download=True,
    transform=transform,
)
loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=0)

def top_activating_images(layer_name, channel, top_k=6, max_batches=3):
    score_chunks = []
    image_chunks = []
    label_chunks = []
    for batch_index, (images, labels) in enumerate(loader):
        if batch_index >= max_batches:
            break
        score_chunks.append(layer_mean_activation(layer_name, channel, images))
        image_chunks.append(images)
        label_chunks.append(labels)
    scores = torch.cat(score_chunks)
    images = torch.cat(image_chunks)
    labels = torch.cat(label_chunks)
    values, indices = torch.topk(scores, k=top_k)
    return images[indices], labels[indices], values

target_layer = "mixed3b"
target_channel = 379
synthetic = render_neuron(target_layer, target_channel, steps=12)
real_images, real_labels, real_scores = top_activating_images(target_layer, target_channel)

fig, axes = plt.subplots(2, 4, figsize=(11, 5.5))
axes[0, 0].imshow(synthetic)
axes[0, 0].axis("off")
axes[0, 0].set_title("Synthetic preference", fontsize=9)
for ax in axes.flatten()[1:]:
    ax.axis("off")

for index in range(6):
    row = index // 3
    col = (index % 3) + 1
    axes[row, col].imshow(denormalize(real_images[index]))
    axes[row, col].axis("off")
    axes[row, col].set_title(
        f"{dataset.classes[int(real_labels[index])]}\nscore={float(real_scores[index]):.3f}",
        fontsize=8,
    )

plt.suptitle("Live validation on real CIFAR-10 images", fontsize=11)
plt.tight_layout()
print("Top CIFAR-10 labels:", [dataset.classes[int(label)] for label in real_labels])


In [ ]:
def make_arc_batch(angle_degrees, image_size=224, radius=60, line_width=5):
    image = Image.new("RGB", (image_size, image_size), (128, 128, 128))
    draw = ImageDraw.Draw(image)
    center_x = image_size // 2
    center_y = image_size // 2
    box = [center_x - radius, center_y - radius, center_x + radius, center_y + radius]
    draw.arc(box, start=float(angle_degrees) - 60.0, end=float(angle_degrees) + 60.0, fill=(255, 255, 255), width=line_width)
    return transform(image).unsqueeze(0), image

angles = np.linspace(0.0, 345.0, 24)
responses = []
preview_angles = [0.0, 60.0, 120.0]

fig = plt.figure(figsize=(12, 3.8))
grid = fig.add_gridspec(1, 4, width_ratios=[1.0, 1.0, 1.0, 1.6])
preview_axes = [fig.add_subplot(grid[0, index]) for index in range(3)]
polar_ax = fig.add_subplot(grid[0, 3], projection="polar")

for ax, angle in zip(preview_axes, preview_angles):
    batch, preview = make_arc_batch(angle)
    ax.imshow(preview)
    ax.axis("off")
    ax.set_title(f"{int(angle)} deg", fontsize=9)

for angle in angles:
    batch, _ = make_arc_batch(angle)
    responses.append(float(layer_mean_activation(target_layer, target_channel, batch)[0]))

responses = np.array(responses)
closed_angles = np.deg2rad(np.append(angles, angles[0]))
closed_responses = np.append(responses, responses[0])
polar_ax.plot(closed_angles, closed_responses, marker="o", color="#c96a28")
polar_ax.set_title("Orientation tuning for mixed3b:379", fontsize=10)
polar_ax.set_theta_zero_location("N")
polar_ax.set_theta_direction(-1)
plt.tight_layout()

preferred_angle = float(angles[int(np.argmax(responses))])
print("Preferred orientation:", round(preferred_angle, 1), "degrees")
print("Peak response:", round(float(responses.max()), 4))


In [ ]:
target_channel = 379
branch_index = target_channel - 320
weights_bottleneck = model.mixed3b_5x5_bottleneck_pre_relu_conv.weight.detach().cpu().squeeze(-1).squeeze(-1)
weights_5x5 = model.mixed3b_5x5_pre_relu_conv.weight.detach().cpu()

target_filter = weights_5x5[branch_index]
bottleneck_importance = target_filter.abs().sum(dim=(1, 2))
upstream_importance = (bottleneck_importance[:, None] * weights_bottleneck.abs()).sum(dim=0)
top_values, top_indices = torch.topk(upstream_importance, k=6)
display_channels = top_indices[:2].tolist()

fig, axes = plt.subplots(1, 4, figsize=(12, 3.6))
for ax, channel in zip(axes[:2], display_channels):
    ax.imshow(render_neuron("mixed3a", int(channel), steps=10))
    ax.axis("off")
    ax.set_title(f"mixed3a:{channel}", fontsize=9)

axes[2].bar(range(len(top_values)), top_values.tolist(), color="#1f5f8b")
axes[2].set_xticks(range(len(top_values)), [str(int(index)) for index in top_indices.tolist()], rotation=30)
axes[2].set_title("Top upstream channels", fontsize=10)
axes[2].set_xlabel("mixed3a channel")

axes[3].imshow(render_neuron("mixed3b", target_channel, steps=10))
axes[3].axis("off")
axes[3].set_title(f"mixed3b:{target_channel}", fontsize=9)
plt.suptitle("Tracing a small circuit through learned weights", fontsize=11)
plt.tight_layout()

print("Top upstream channels for mixed3b:379:")
for channel, value in zip(top_indices.tolist(), top_values.tolist()):
    print(f"  mixed3a:{int(channel):>3} -> importance {float(value):.3f}")


## Takeaway

This notebook now earns the basic visual-circuits story live: a neuron preference, a real-image check, a tuning curve, and an upstream-weight trace. The picture is still cleaner than language-model interpretability, but it is no longer a slideshow.


## Turn this notebook into research output

Research worksheet means this notebook is not complete when the cells finish. Use the templates in /templates and leave behind written outputs.


### Before you run

- Name the three terms you want to leave with: feature, circuit, intervention.
- Predict which live section will change your intuition most: feature visualization, CIFAR validation, orientation tuning, or circuit tracing.
- Open the paper-reading-note template before you run the notebook.


### After you run

- Write one paragraph on which output was actually measured live and which claims still rely on interpretation.
- List the first ambiguity that appears when you try to generalize this visual result to language models.


### Ship these artifacts

- One reading note with a mentor/peer-summary paragraph.
- One experiment log covering at least one measured activation sweep.
- One glossary list of unclear terms.
- One next-question list for M01.


## Self-check questions

- Without looking back at the note, explain what feature, circuit, and intervention mean and how they relate to one another.
- Why is the Zoom In visual-circuits story so useful for beginners, yet risky to transfer directly to language models?
- If M00 is the starting picture for later interpretability work, which part breaks first: monosemantic neurons, directly visible local circuits, or causal explanation? Why?
- If you cannot answer at least two of these without rereading the note, revisit the article and your written outputs.
